<a href="https://colab.research.google.com/github/AhmedEsammohmaed/IBS/blob/main/Sttreamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q streamlit ezdxf pdf2image PyPDF2 albumentations segmentation-models-pytorch
!apt-get install -y poppler-utils
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 70.1 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 1 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.13 [186 kB]
Fetched 186 kB in 1s (324 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 118419 files and directories currently installed.)
Preparing

In [4]:
import streamlit as st
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏
changed 22 packages in 1s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏

In [3]:
%%writefile app.py
import io
import json
import os
import numpy as np
import cv2
import torch
import torch.nn as nn
import streamlit as st
from PIL import Image
import ezdxf
import segmentation_models_pytorch as smp

# --- 1. CONFIG & CONSTANTS ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 512
NUM_ROOM_CLASSES = 12
NUM_ICON_CLASSES = 11

ROOM_CLASSES = [
    "Background", "Outdoor", "Wall", "Kitchen", "Living Room",
    "Bed Room", "Bath", "Hallway", "Railing", "Storage", "Garage", "Other"
]

ICON_CLASSES = [
    "Background", "Window", "Door", "Closet", "Electrical Appliance",
    "Toilet", "Sink", "Sauna/Bench", "Chimney", "Bathtub", "Counter"
]

COLOR_MAP = np.array([
    [0, 0, 0],       # Background
    [200, 200, 200], # Outdoor
    [50, 50, 50],    # Wall
    [255, 179, 0],   # Kitchen
    [33, 150, 243],  # Living Room
    [156, 39, 176],  # Bed Room
    [0, 188, 212],   # Bath
    [139, 195, 74],  # Hallway
    [255, 87, 34],   # Railing
    [121, 85, 72],   # Storage
    [96, 125, 139],  # Garage
    [158, 158, 158]  # Other
], dtype=np.uint8)


# --- 2. MODEL ARCHITECTURE ---
class MultiTaskUNet(nn.Module):
    def __init__(self, n_room_cls=NUM_ROOM_CLASSES, n_icon_cls=NUM_ICON_CLASSES):
        super().__init__()
        self.base = smp.Unet(
            encoder_name="resnet34",
            encoder_weights=None,
            in_channels=3,
            classes=n_room_cls
        )
        self.encoder = self.base.encoder
        self.decoder = self.base.decoder
        self.room_head = self.base.segmentation_head

        in_ch = self.base.decoder.blocks[-1].conv2[0].out_channels
        self.icon_head = nn.Sequential(
            nn.Conv2d(in_ch, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, n_icon_cls, 1)
        )

    def forward(self, x):
        features = self.encoder(x)
        decoder_out = self.decoder(*features)
        room_logits = self.room_head(decoder_out)
        icon_logits = self.icon_head(decoder_out)
        return room_logits, icon_logits


@st.cache_resource
def load_model(weights_path=None):
    model = MultiTaskUNet().to(DEVICE)
    if weights_path and os.path.exists(weights_path):
        try:
            checkpoint = torch.load(weights_path, map_location=DEVICE)

            # Handle case where full model was saved instead of state_dict
            if isinstance(checkpoint, torch.nn.Module):
                state_dict = checkpoint.state_dict()
            elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
                state_dict = checkpoint["state_dict"]
            else:
                state_dict = checkpoint

            # Strip 'module.' or unexpected prefixes if trained with DataParallel
            new_state_dict = {}
            for k, v in state_dict.items():
                name = k.replace("module.", "")
                new_state_dict[name] = v

            # Load with strict=False to bypass non-critical missing/extra keys
            model.load_state_dict(new_state_dict, strict=False)
            st.toast("Model weights loaded successfully!", icon="✅")
        except Exception as e:
            st.warning(f"Could not load weights from '{weights_path}': {e}")
    model.eval()
    return model

# --- 3. INFERENCE & PROCESSING HEURISTICS ---
def preprocess_image(pil_img, img_size=IMG_SIZE):
    orig_w, orig_h = pil_img.size
    img_rgb = np.array(pil_img.convert("RGB"))
    resized = cv2.resize(img_rgb, (img_size, img_size))
    norm = resized.astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    norm = (norm - mean) / std
    tensor = torch.tensor(norm).permute(2, 0, 1).unsqueeze(0).float()
    return tensor.to(DEVICE), img_rgb, (orig_w, orig_h)


def run_pipeline(model, tensor_img, orig_shape, px_per_m=50.0):
    orig_w, orig_h = orig_shape
    with torch.no_grad():
        r_logits, i_logits = model(tensor_img)
        r_pred = torch.argmax(r_logits, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
        i_pred = torch.argmax(i_logits, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

    r_mask = cv2.resize(r_pred, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
    i_mask = cv2.resize(i_pred, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)

    rooms = []
    non_room_classes = {0, 1, 2}
    floor_mask = np.isin(r_mask, list(non_room_classes), invert=True).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    opened = cv2.morphologyEx(floor_mask, cv2.MORPH_OPEN, kernel)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(opened, connectivity=4)

    for label_id in range(1, num_labels):
        area_px = stats[label_id, cv2.CC_STAT_AREA]
        if area_px < 200:
            continue

        region_mask = (labels == label_id).astype(np.uint8)
        vals, counts = np.unique(r_mask[region_mask == 1], return_counts=True)
        assigned_cls_id = vals[np.argmax(counts)]
        area_m2 = area_px / (px_per_m ** 2)

        contours, _ = cv2.findContours(region_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        poly_pts = []
        if contours:
            epsilon = 0.015 * cv2.arcLength(contours[0], True)
            approx = cv2.approxPolyDP(contours[0], epsilon, True)
            poly_pts = approx.squeeze().tolist() if len(approx) >= 3 else []

        rooms.append({
            "room_id": int(label_id),
            "type": ROOM_CLASSES[assigned_cls_id],
            "area_m2": round(float(area_m2), 2),
            "polygon": poly_pts
        })

    icons = []
    for cls_idx in range(1, NUM_ICON_CLASSES):
        bin_icon = (i_mask == cls_idx).astype(np.uint8)
        if not np.any(bin_icon):
            continue
        n_lbls, _, stats, _ = cv2.connectedComponentsWithStats(bin_icon)
        for i in range(1, n_lbls):
            x, y, w, h, a = stats[i]
            if a > 20:
                icons.append({
                    "class": ICON_CLASSES[cls_idx],
                    "bbox": [int(x), int(y), int(w), int(h)]
                })

    return r_mask, i_mask, rooms, icons


# --- 4. EXPORT UTILITIES ---
def export_to_dxf(rooms, icons):
    doc = ezdxf.new("R2010")
    msp = doc.modelspace()

    for r in rooms:
        pts = r["polygon"]
        if len(pts) >= 3:
            dxf_pts = [(p[0], -p[1]) for p in pts]
            msp.add_lwpolyline(dxf_pts, close=True, dxfattribs={"layer": "ROOMS"})
            cx = sum(p[0] for p in pts) / len(pts)
            cy = -sum(p[1] for p in pts) / len(pts)
            label = f"{r['type']}\n{r['area_m2']} m2"
            msp.add_text(label, dxfattribs={"height": 12, "layer": "LABELS"}).set_placement((cx, cy))

    for ic in icons:
        x, y, w, h = ic["bbox"]
        pts = [(x, -y), (x + w, -y), (x + w, -(y + h)), (x, -(y + h))]
        msp.add_lwpolyline(pts, close=True, dxfattribs={"layer": "ICONS"})

    buf = io.StringIO()
    doc.write(buf)
    return buf.getvalue()


# --- 5. STREAMLIT UI ---
st.set_page_config(page_title="Floor Plan Vectorizer AI", layout="wide")

st.title("🏗️ AI Floor Plan Multi-Task Analysis & Vectorization")
st.write("Upload a floor plan image (or PDF) to run automated room segmentation, icon detection, area estimation, and CAD export.")

st.sidebar.header("⚙️ Configuration")
model_weight_file = st.sidebar.file_uploader("Upload Trained Model Weights (.pt)", type=["pt", "pth"])
px_per_m = st.sidebar.number_input("Pixels per Meter (Scale Ratio)", min_value=1.0, max_value=200.0, value=50.0)

weights_path = "cubicasa_multitask.pt"
if model_weight_file is not None:
    weights_path = "temp_weights.pt"
    with open(weights_path, "wb") as f:
        f.write(model_weight_file.getbuffer())

model = load_model(weights_path)

uploaded_file = st.file_uploader("Upload Floor Plan Image or PDF", type=["png", "jpg", "jpeg", "pdf"])

if uploaded_file is not None:
    pil_image = None
    if uploaded_file.type == "application/pdf":
        try:
            from pdf2image import convert_from_bytes
            images = convert_from_bytes(uploaded_file.read())
            pil_image = images[0]
            st.info("Loaded Page 1 from PDF.")
        except Exception as e:
            st.error(f"Failed to read PDF: {e}. Ensure poppler-utils is installed.")
    else:
        pil_image = Image.open(uploaded_file)

    if pil_image is not None:
        col1, col2 = st.columns(2)
        with col1:
            st.subheader("Original Input")
            st.image(pil_image, use_container_width=True)

        with st.spinner("Processing floor plan through Multi-Task U-Net pipeline..."):
            tensor_img, orig_rgb, orig_shape = preprocess_image(pil_image)
            r_mask, i_mask, rooms, icons = run_pipeline(model, tensor_img, orig_shape, px_per_m)

            colored_segmentation = COLOR_MAP[r_mask]
            blended = cv2.addWeighted(orig_rgb, 0.6, colored_segmentation, 0.4, 0)

            annotated = orig_rgb.copy()
            for ic in icons:
                x, y, w, h = ic["bbox"]
                cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 255, 0), 2)
                cv2.putText(annotated, ic["class"], (x, max(15, y - 5)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        with col2:
            st.subheader("Semantic Room Segmentation")
            st.image(blended, use_container_width=True)

        st.subheader("Detected Architectural Icons")
        st.image(annotated, use_container_width=True)

        st.write("---")
        st.header("📊 Analytical Output & Exports")

        tab1, tab2, tab3 = st.tabs(["🏡 Room Breakdown", "🛋️ Detected Icons", "💾 CAD & Data Export"])

        with tab1:
            st.subheader("Extracted Rooms & Calculated Areas")
            if rooms:
                st.dataframe(rooms, use_container_width=True)
            else:
                st.warning("No room boundaries isolated based on current segmentation.")

        with tab2:
            st.subheader("Icon Bounding Boxes")
            if icons:
                st.dataframe(icons, use_container_width=True)
            else:
                st.info("No architectural icons detected.")

        with tab3:
            st.subheader("Download Structured Outputs")

            output_json = {
                "rooms": rooms,
                "icons": icons,
                "metadata": {"px_per_m": px_per_m, "image_size": orig_shape}
            }
            json_str = json.dumps(output_json, indent=2)

            st.download_button(
                label="📄 Download Floor Plan JSON Data",
                data=json_str,
                file_name="floor_plan_analysis.json",
                mime="application/json"
            )

            try:
                dxf_data = export_to_dxf(rooms, icons)
                st.download_button(
                    label="📐 Download CAD DXF Vector File",
                    data=dxf_data,
                    file_name="floor_plan_vector.dxf",
                    mime="image/vnd.dxf"
                )
            except Exception as e:
                st.error(f"Error generating DXF file: {e}")

Writing app.py


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙

⠹⠸⠼⠴⠦⠧2026-09-01 22:53:14.716 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://8.229.79.72:8501

your url is: https://shiny-sloths-heal.loca.lt
